# 카이제곱 검정

## 0. 카이제곰 검정 기초 개념
- 카이제곱 검정(Chi-square test)은 범주형 데이터의 관찰된 빈도(observed)와 기대되는 빈도(expected)를 비교해 두 변수 간의 독립성이나 분포의 적합성을 검정하는 방법

| 유형 | 목적 | 변수 개수 |
|---|---|---|
| 적합도 검정 | 1개의 범주형 변수가 특정 분포(비율)를 따르는지 확인 | 1개 |
| 독립성 검정 | 서로 다른 2개의 범주형 변수가 서로 관련이 있는지(독립인지)확인 | 2개 |
| 동질성 검정 | 여러 집단(그룹)의 분포가 동일한지 확인 | 2개(그룹 변수 포함) |

> 독립성 검정과 동질성 검정은 "계산 방법(사용 함수)이 완전히 동일"하다.
차이는 "무엇을 묻는 질문인가"에 따라 다른다.
    - 독립성 검정 : 두 변수가 서로 관련 있는가?
    - 동질성 검정 : 여러 집단의 분포가 같은가?


## 1. 적합도 검정

- 귀무가설 : 특정 분포(비율)를 따른다.
- 대립가설 : 특정 분포(비율)를 따르지 않는다.

---

`scipy.stats.chisquare(observed, expected, ddof, axis)`

| 파라미터 | 설명 |
|---|---|
| `observed` | 관측된 빈도 리스트(배열) |
| `expected` | 기대 빈도 리스트(배열), 생략 시 모든 카테고리의 빈도가 균일하다고 가정 |
| `ddof` | 자유도 조정값, 기본값 0 |
| `axis` | 축, 기본값 0 |

In [1]:
from scipy import stats

# 실제 관측된 빈도 (예 - 바닐라 150, 초코 120, 딸기 30)
observed = [150, 120, 30]

# 이론적으로 기대되는 빈도
# 전체 300개 - 바닐라 50 %, 초코 35&, 딸기 15%
expected = [0.5*300, 0.35*300, 0.15*300]

# chisquare(관측값, 기대값)
stats.chisquare(observed, expected) # p-value가 0.025 유의수전 0.05보다 작다!
# --> 귀무가설 기각, 대립가설 채택 --> 이 도시의 아이스크림 맛 선호도는 전국적인 맛 선호도와 다르다!

Power_divergenceResult(statistic=7.142857142857142, pvalue=0.028115659748972056)

## 2. 독립성 검정
- 서로 다른 두 범주형 변수가 서로 관련이 있는지(독립적인지) 검정

- 귀무가설 : 두 범주형 변수가 독립적이다(서로 연관성이 없다.)
- 대립가설 : 두 범주형 변수가 독립적이지 않다(서로 연관선이 있다.)

---
`scipy.stats.chi2_contingencey(table, correction=True)`
| 파라미터 | 설명 |
|---|---|
| `table` | 교차표 데이터(contingency table), 2차원형태|
| `correction` | 연속성 보정 적용 여부, 기분값 True, 문제에서 "연속성 수정을 하지 않는다"는 조건이 없으면 기본값 유지 있으면 False로 설정 |

반환되는 expected_freq(기대 빈도) : 두 변수가 독립적이라고 가정했을 때 이론적으로 기대되는 값
    - 교차표를 좌->우, 위->아래 순서로 읽으면서 기대빈도 배열도 같은 순서로 1:1 매칭해서 해석한다.

In [2]:
# 방법 1 - 데이터프레임 만들기 (컬럼(열) 방향)
import pandas as pd
df = pd.DataFrame({
    '좋아함' : [89, 90],
    '좋아하지 않음' : [30,10]},
    index=['남자', '여자']
)

df

,좋아함,좋아하지 않음
남자,89,30
여자,90,10


In [3]:
from scipy import stats
stats.chi2_contingency(df)
# 카이제곱 통계량, p-value, 자유도, 기대빈도표 반환
# p-value 가 0.05(유의수준)보다 작다. --> 두 변수(성별, 운동선호)는 서로 독립적이다.(서로 관린이 있다.)

Chi2ContingencyResult(statistic=7.432542542016808, pvalue=0.006405477551514457, dof=1, expected_freq=array([[97.26484018, 21.73515982],
       [81.73515982, 18.26484018]]))

In [4]:
# raw(로우) 데이터에서 교차표 만들어서 검정
import pandas as pd
data = {
    '성별': ['남자'] * 110 + ['여자'] * 100,
    '운동': ['좋아함'] * 80 + ['좋아하지 않음'] * 30 + ['좋아함'] * 90 + ['좋아하지 않음'] * 10
}

df = pd.DataFrame(data)
df

,성별,운동
0,남자,좋아함
1,남자,좋아함
2,남자,좋아함
3,남자,좋아함
4,남자,좋아함
...,...,...
205,여자,좋아하지 않음
206,여자,좋아하지 않음
207,여자,좋아하지 않음
208,여자,좋아하지 않음


In [5]:
# pd.crosstab(행 기준 컬럼, 열 기준 컬럼) : 로우(raw) 데이터를 자동으로 교차표로 변환
df = pd.crosstab(df['성별'], df['운동'])
df

운동,좋아하지 않음,좋아함
성별,,
남자,30,80
여자,10,90


## 3. 동질성 검정
- 서로 다른 집단(그룹)들의 분포가 동일한지 검정
- 여러 집단의 분포가 같은가
- 사용하는 함수는 독립성 검정과 같다.

In [8]:
# 교차표 기반 카이제곱 검정
import pandas as pd
from scipy import stats
# 이미 집계뙨 2x2 교차표
df = pd.DataFrame([
    [50, 50],   # 통계학과
    [30, 70]    # 컴퓨터공학과
])

stats.chi2_contingency(df)
# p-value가 유의수준(0.05)보다 작으므로 귀무가설을 기각하고 대립가설을 채택
# 두 학과의 동아리 가입 비율은 동일하지 않다.

Chi2ContingencyResult(statistic=7.520833333333334, pvalue=0.006098945931214352, dof=1, expected_freq=array([[40., 60.],
       [40., 60.]]))

In [20]:
# 로우 데이터가 주어졌을 때
import pandas as pd
data = {
    '학과' : ['통계학과'] * 100 + ['컴퓨터공학과'] * 100,
    '동아리가입여부' : ['가입'] * 50 + ['미가입'] * 50 + ['가입'] * 30 + ['미가입'] * 70
}
df= pd.DataFrame(data)
df.sample(5)    # 무작위 5개의 행 확인

,학과,동아리가입여부
73,통계학과,미가입
11,통계학과,가입
40,통계학과,가입
171,컴퓨터공학과,미가입
195,컴퓨터공학과,미가입


In [21]:
df = pd.crosstab(df['학과'], df['동아리가입여부'])
df

동아리가입여부,가입,미가입
학과,,
컴퓨터공학과,30,70
통계학과,50,50


In [22]:
from scipy import stats
stats.chi2_contingency(df)

Chi2ContingencyResult(statistic=7.520833333333334, pvalue=0.006098945931214352, dof=1, expected_freq=array([[40., 60.],
       [40., 60.]]))